<a href="https://colab.research.google.com/github/edwardsnj/genomic-llms-in-practice/blob/main/Workshop/Exercise_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Workshop: Fine-tuning TinyDNABERT for DNA sequence classification tasks
Model: https://huggingface.co/ChengsenWang/TinyDNABERT-20M-V1

Dataset: https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised

1. Load necessary packages

In [ ]:
import torch
from typing import Any, Tuple, Union
import tensorflow as tf
import numpy as np
import pandas as pd
import scipy as sp

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    EvalPrediction,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax

from peft import get_peft_model, LoraConfig, TaskType

2. Load the dataset

In [ ]:
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
print(dataset)

  a. Separate the train and test datasets.

In [ ]:
# Train dataset
train_dataset = dataset["train"]

# Evaluation dataset
test_dataset = dataset["test"]

  b. Filter only for the promoter classification task. This is a binary classification task which involves categorising each sample as "promoter" (1) or "not promoter" (0).

  *Hint: use the `filter` function*

In [ ]:
# print(set(train_dataset.to_pandas()["task"]))
train_dataset = train_dataset.filter(lambda example: example["task"] == "promoter_all")
test_dataset = test_dataset.filter(lambda example: example["task"] == "promoter_all")

b. Split the train dataset further into train (80%) and validation (20%).

*Hint: use the `train_test_split` function.*

In [ ]:
# Split the training data into training and validation
split_dataset = train_dataset.train_test_split(test_size=0.2)
print(split_dataset)

In [ ]:
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [ ]:
print(f"No. train samples: {len(train_dataset)}\nNo. val. samples: {len(val_dataset)}\nNo. test samples: {len(test_dataset)}")

d. To make the training less time-consuming, we will use only the first 10000 rows of the training data.

*Hint: use the `select` function.*

In [ ]:
train_dataset = train_dataset.shuffle(seed=42).select(range(10000))

3. Load the tokeniser and define the `preprocess_function`, which we will use to batch apply tokenisation to our datasets.

In [ ]:
# Set the model name from the Huggingface page
MODEL_NAME = "ChengsenWang/TinyDNABERT-20M-V1"

In [ ]:
# Load the tokeniser
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [ ]:
# Define the preprocessing function
def preprocess_function(examples):
    return tokeniser(
        examples["sequence"],
        truncation=True,
        padding="max_length",
        max_length=512,
        # YOUR CODE HERE: SET THE MAXIMUM LENGTH AS 512
        # YOUR CODE HERE: PAD THE TOKENISED SEQUENCE TO THE MAXIMUM LENGTH
    )

4. Tokenise the data

*Hint: use the `map` function*

In [ ]:
train_ds_encoded = train_dataset.map(preprocess_function,batched=True)
val_ds_encoded = val_dataset.map(preprocess_function,batched=True)
test_ds_encoded = test_dataset.map(preprocess_function,batched=True)

5. Fine-tune for the promoter classification task

In [ ]:
# Load the pretrained weights and add a classification head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

In [ ]:
# Select the training arguments
training_args = TrainingArguments(
    output_dir="tinydnabert-finetuned",
    # YOUR CODE HERE: TRAIN THE MODEL FOR 3 EPOCHS
    num_train_epochs=3,
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 16
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    # YOUR CODE HERE: SET THE LEARNING RATE TO 2E-5
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
# Set up the trainer with the training arguments above
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=val_ds_encoded
    )

In [ ]:
trainer.train()
trainer.save_model("./tinydnabert-finetuned")
tokeniser.save_pretrained("./tinydnabert-finetuned")
trainer.evaluate()

#### Evaluate on promoter classification

In [ ]:
# TrainingArguments for evaluation only
training_args_eval= TrainingArguments(
    output_dir="./results",
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 24
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    # YOUR CODE HERE: SET UP SO THAT WE DO EVALUATION BUT NOT TRAINING
    do_train=False,
    logging_dir="./logs",
    report_to="none",
)

In [ ]:
# Define metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    # Handle output as tuple (logits,) or ModelOutput
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = logits.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    probabilities = softmax(logits, axis=-1)[:, 1] # probability for class 1
    auroc = roc_auc_score(labels, probabilities)  # using sklearn
    return {"accuracy": acc, "auroc": auroc, "f1": f1}

In [ ]:
trainer_eval = Trainer(
    model=model,
    args=training_args_eval,
    eval_dataset=test_ds_encoded,
    compute_metrics=compute_metrics,
)

In [ ]:
# Evaluate model
eval_results = trainer_eval.evaluate()
print(eval_results)

shuffle = test_dataset.shuffle(seed=42)[:100]
print(shuffle["label"])
preds = []
for seq in shuffle["sequence"]:
  enc = tokeniser(
        seq,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
  ).to("cuda")
  with torch.no_grad():
    out = model(**enc)
  preds.append(out.logits.to("cpu")[0].argmax().item())
print(preds)
